# Youth Unemployment Explorer — ACC102 Track 4
**Analytical Notebook: Global Youth Unemployment Trends (1991–2022)**

---

## Problem Definition

**Analytical Question:** How has youth unemployment evolved globally between 1991 and 2022, and what structural and economic factors are associated with higher rates?

**Target User / Audience:** Policy analysts, economics students, and journalists who need an accessible, interactive overview of global youth labour market trends without requiring technical programming skills.

**Product:** An interactive Streamlit dashboard (Track 4) that allows users to explore youth unemployment trends by country, region, year range, and economic structure.

---

## Data Sources

| Dataset | Source | Accessed |
|---|---|---|
| `youth_unemployment_global.csv` | World Bank / ILO via Kaggle | April 2025 |
| `Employment_Unemployment_GDP_data.csv` | World Bank Open Data via Kaggle | April 2025 |

Both datasets are publicly available under open-data licences for educational use.

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f5f9ff',
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 1.2,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Libraries loaded successfully.')

## Step 2 — Data Loading

Both CSV files are loaded from the same directory as this notebook (relative paths).

In [ ]:
# Load raw datasets
youth_raw = pd.read_csv('youth_unemployment_global.csv')
gdp_raw   = pd.read_csv('Employment_Unemployment_GDP_data.csv')

print('youth_unemployment_global.csv shape:', youth_raw.shape)
print('Employment_Unemployment_GDP_data.csv shape:', gdp_raw.shape)
print()
print('Youth dataset columns:', list(youth_raw.columns))
print('GDP dataset columns:  ', list(gdp_raw.columns))

In [ ]:
# Preview youth dataset
youth_raw.head()

In [ ]:
# Preview GDP / employment dataset
gdp_raw.head()

## Step 3 — Data Cleaning and Preparation

Key steps:
- Drop rows with missing `YouthUnemployment` values
- Filter to the analysis window 1991–2022
- Merge both datasets on `Country` and `Year`
- Engineer derived features: GDP in billions, log-GDP, youth–adult gap, dominant employment sector
- Assign regions manually based on country names

In [ ]:
# --- Cleaning: youth dataset ---
print('Missing YouthUnemployment before drop:', youth_raw['YouthUnemployment'].isna().sum())

youth = youth_raw.dropna(subset=['YouthUnemployment'])
youth = youth[youth['Year'].between(1991, 2022)]

print('Rows after cleaning:', len(youth))
print('Year range:', youth['Year'].min(), '–', youth['Year'].max())
print('Unique countries:', youth['Country'].nunique())

In [ ]:
# --- Missing values in GDP dataset ---
print('Missing values per column in GDP dataset:')
print(gdp_raw.isna().sum())

In [ ]:
# --- Merge datasets ---
# Join on Country + Year (youth uses 'Country', GDP uses 'Country Name')
df = youth.merge(
    gdp_raw,
    left_on=['Country', 'Year'],
    right_on=['Country Name', 'Year'],
    how='inner'
)

print('Merged dataframe shape:', df.shape)
print('Countries in merged dataset:', df['Country'].nunique())

In [ ]:
# --- Feature Engineering ---

# GDP in billions (easier to read)
df['GDP_bn'] = df['GDP (in USD)'] / 1e9

# Log-GDP for scatter plot (reduces skew from large outliers like USA/China)
df['GDP_log'] = np.log10(df['GDP (in USD)'])

# Youth–adult unemployment gap: how much higher is youth unemployment vs total?
df['Youth_Gap'] = df['YouthUnemployment'] - df['Unemployment Rate']

# Dominant employment sector per country-year
sector_cols = ['Employment Sector: Agriculture',
               'Employment Sector: Industry',
               'Employment Sector: Services']
df['Dom_Sector'] = (
    df[sector_cols]
    .idxmax(axis=1)
    .str.replace('Employment Sector: ', '', regex=False)
)

print('New columns added:', ['GDP_bn', 'GDP_log', 'Youth_Gap', 'Dom_Sector'])
print()
print('Youth_Gap summary statistics:')
print(df['Youth_Gap'].describe().round(2))

In [ ]:
# --- Region Mapping ---
# Manually map each country to a world region for regional analysis

region_map = {
    'Afghanistan':'South Asia','Bangladesh':'South Asia','India':'South Asia',
    'Nepal':'South Asia','Pakistan':'South Asia','Sri Lanka':'South Asia',
    'Albania':'Eastern Europe','Armenia':'Eastern Europe','Azerbaijan':'Eastern Europe',
    'Belarus':'Eastern Europe','Bosnia and Herzegovina':'Eastern Europe',
    'Bulgaria':'Eastern Europe','Croatia':'Eastern Europe','Czech Republic':'Eastern Europe',
    'Estonia':'Eastern Europe','Georgia':'Eastern Europe','Hungary':'Eastern Europe',
    'Kazakhstan':'Eastern Europe','Kosovo':'Eastern Europe','Latvia':'Eastern Europe',
    'Lithuania':'Eastern Europe','Moldova':'Eastern Europe','Montenegro':'Eastern Europe',
    'North Macedonia':'Eastern Europe','Poland':'Eastern Europe','Romania':'Eastern Europe',
    'Russia':'Eastern Europe','Serbia':'Eastern Europe','Slovak Republic':'Eastern Europe',
    'Slovenia':'Eastern Europe','Tajikistan':'Eastern Europe','Turkey':'Eastern Europe',
    'Turkmenistan':'Eastern Europe','Ukraine':'Eastern Europe','Uzbekistan':'Eastern Europe',
    'Australia':'Asia-Pacific','China':'Asia-Pacific','Indonesia':'Asia-Pacific',
    'Japan':'Asia-Pacific','Korea, Rep.':'Asia-Pacific','Malaysia':'Asia-Pacific',
    'Myanmar':'Asia-Pacific','New Zealand':'Asia-Pacific','Philippines':'Asia-Pacific',
    'Thailand':'Asia-Pacific','Vietnam':'Asia-Pacific','Cambodia':'Asia-Pacific',
    'Austria':'Western Europe','Belgium':'Western Europe','Denmark':'Western Europe',
    'Finland':'Western Europe','France':'Western Europe','Germany':'Western Europe',
    'Greece':'Western Europe','Iceland':'Western Europe','Ireland':'Western Europe',
    'Italy':'Western Europe','Luxembourg':'Western Europe','Netherlands':'Western Europe',
    'Norway':'Western Europe','Portugal':'Western Europe','Spain':'Western Europe',
    'Sweden':'Western Europe','Switzerland':'Western Europe','United Kingdom':'Western Europe',
    'Algeria':'Middle East & Africa','Angola':'Middle East & Africa',
    'Bahrain':'Middle East & Africa','Egypt, Arab Rep.':'Middle East & Africa',
    'Ethiopia':'Middle East & Africa','Ghana':'Middle East & Africa',
    'Iran, Islamic Rep.':'Middle East & Africa','Iraq':'Middle East & Africa',
    'Jordan':'Middle East & Africa','Kenya':'Middle East & Africa',
    'Kuwait':'Middle East & Africa','Lebanon':'Middle East & Africa',
    'Morocco':'Middle East & Africa','Nigeria':'Middle East & Africa',
    'Saudi Arabia':'Middle East & Africa','South Africa':'Middle East & Africa',
    'Tunisia':'Middle East & Africa','United Arab Emirates':'Middle East & Africa',
    'Yemen, Rep.':'Middle East & Africa',
    'Argentina':'Latin America','Bolivia':'Latin America','Brazil':'Latin America',
    'Chile':'Latin America','Colombia':'Latin America','Costa Rica':'Latin America',
    'Ecuador':'Latin America','Mexico':'Latin America','Peru':'Latin America',
    'Uruguay':'Latin America','Venezuela, RB':'Latin America',
    'Canada':'North America','United States':'North America',
}

df['Region'] = df['Country'].map(region_map).fillna('Other')
df = df.sort_values(['Country', 'Year']).reset_index(drop=True)

print('Region distribution:')
print(df['Region'].value_counts())

In [ ]:
# --- Final dataset overview ---
print('Final dataset shape:', df.shape)
print()
df.head()

## Step 4 — Descriptive Analysis

### 4.1 Global Summary Statistics

In [ ]:
# Summary statistics for key variables
summary_cols = ['YouthUnemployment', 'Unemployment Rate', 'Youth_Gap', 'GDP_bn']
df[summary_cols].describe().round(2)

In [ ]:
# Top 10 highest youth unemployment country-years on record
df.nlargest(10, 'YouthUnemployment')[['Country', 'Year', 'YouthUnemployment', 'Region']].reset_index(drop=True)

In [ ]:
# Average youth unemployment by region (full period)
df.groupby('Region')['YouthUnemployment'].agg(['mean', 'median', 'min', 'max']).round(2).sort_values('mean', ascending=False)

## Step 5 — Analysis and Visualisation

### 5.1 Global Trend: Youth vs Total Unemployment (1991–2022)

In [ ]:
trend = (df.groupby('Year')['YouthUnemployment'].mean().reset_index()
           .rename(columns={'YouthUnemployment': 'AvgYouth'}))
trend_tot = (df.groupby('Year')['Unemployment Rate'].mean().reset_index()
               .rename(columns={'Unemployment Rate': 'AvgAll'}))
trend = trend.merge(trend_tot, on='Year')

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between(trend['Year'], trend['AvgYouth'], alpha=0.12, color='#1565c0')
ax.plot(trend['Year'], trend['AvgYouth'], color='#1565c0', lw=2.5,
        marker='o', ms=4, label='Youth unemployment (avg)')
ax.plot(trend['Year'], trend['AvgAll'],  color='#e53935', lw=2,
        ls='--', marker='s', ms=3, label='Total unemployment (avg)')
ax.axvspan(2008, 2010, alpha=0.12, color='grey', label='Global financial crisis')
ax.axvspan(2019.5, 2021, alpha=0.12, color='orange', label='COVID-19')
ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Unemployment Rate (%)', fontsize=10)
ax.set_title('Global Average Youth vs Total Unemployment (1991–2022)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig1_global_trend.png', dpi=150, bbox_inches='tight')
plt.show()

peak_row = trend.loc[trend['AvgYouth'].idxmax()]
avg_gap_global = (trend['AvgYouth'] - trend['AvgAll']).mean()
print(f"Peak youth unemployment: {peak_row['AvgYouth']:.1f}% in {int(peak_row['Year'])}")
print(f"Average youth-adult gap across full period: {avg_gap_global:.1f} percentage points")

**Insight:** Youth unemployment consistently runs roughly 10–12 percentage points above the total rate. Both the 2008–09 financial crisis and the COVID-19 shock produced sharp spikes, with youth workers disproportionately affected in both episodes.

### 5.2 Regional Breakdown — Latest Year (2022)

In [ ]:
REGION_COLOURS = {
    'Western Europe':       '#1565c0',
    'Eastern Europe':       '#2e7d32',
    'Latin America':        '#c62828',
    'Asia-Pacific':         '#e65100',
    'Middle East & Africa': '#6a1b9a',
    'South Asia':           '#4e342e',
    'North America':        '#00838f',
    'Other':                '#9e9e9e',
}

latest = df[df['Year'] == 2022].copy()
regions_to_show = [r for r in REGION_COLOURS if r != 'Other']
latest_filt = latest[latest['Region'].isin(regions_to_show)]

region_order = (latest_filt.groupby('Region')['YouthUnemployment']
                .median().sort_values(ascending=False).index.tolist())

fig2, ax2 = plt.subplots(figsize=(9, 5))
sns.boxplot(data=latest_filt, y='Region', x='YouthUnemployment',
            order=region_order, palette=REGION_COLOURS,
            width=0.5, linewidth=1.2, ax=ax2)
ax2.set_xlabel('Youth Unemployment Rate (%)', fontsize=10)
ax2.set_ylabel('')
ax2.set_title('Youth Unemployment by Region (2022)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_regional_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

print('Median youth unemployment by region (2022):')
print(latest_filt.groupby('Region')['YouthUnemployment'].median().sort_values(ascending=False).round(1))

### 5.3 Regional Trends Over Time

In [ ]:
reg_trend = (df[df['Region'].isin(regions_to_show)]
             .groupby(['Year', 'Region'])['YouthUnemployment'].mean().reset_index())

fig3, ax3 = plt.subplots(figsize=(12, 5))
for reg in regions_to_show:
    sub = reg_trend[reg_trend['Region'] == reg]
    if not sub.empty:
        ax3.plot(sub['Year'], sub['YouthUnemployment'],
                 label=reg, color=REGION_COLOURS.get(reg, '#999'),
                 lw=1.8, marker='o', ms=2.5)
ax3.set_xlabel('Year', fontsize=10)
ax3.set_ylabel('Avg Youth Unemployment (%)', fontsize=10)
ax3.set_title('Regional Average Youth Unemployment Over Time (1991–2022)', fontsize=12, fontweight='bold')
ax3.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('fig3_regional_trends.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 GDP vs Youth Unemployment (Correlation Analysis)

In [ ]:
scatter_data = latest_filt.dropna(subset=['GDP_log'])

fig4, ax4 = plt.subplots(figsize=(9, 5))
for reg, grp in scatter_data.groupby('Region'):
    ax4.scatter(grp['GDP_log'], grp['YouthUnemployment'],
                label=reg, color=REGION_COLOURS.get(reg, '#999'),
                alpha=0.75, s=60, edgecolors='white', lw=0.5)

# OLS trend line
x_vals = scatter_data['GDP_log'].values
y_vals = scatter_data['YouthUnemployment'].values
z = np.polyfit(x_vals, y_vals, 1)
p = np.poly1d(z)
xs = np.linspace(x_vals.min(), x_vals.max(), 100)
ax4.plot(xs, p(xs), 'k--', lw=1.2, alpha=0.5, label='Trend line')

r = np.corrcoef(x_vals, y_vals)[0, 1]
ax4.text(0.05, 0.95, f'Pearson r = {r:.2f}', transform=ax4.transAxes,
         fontsize=10, va='top', color='#333')
ax4.set_xlabel('log₁₀ GDP (USD)', fontsize=10)
ax4.set_ylabel('Youth Unemployment (%)', fontsize=10)
ax4.set_title('GDP vs Youth Unemployment — Cross-Country (2022)', fontsize=12, fontweight='bold')
ax4.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('fig4_gdp_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pearson correlation (log GDP vs youth unemployment): r = {r:.3f}')
strength = 'weak' if abs(r) < 0.3 else ('moderate' if abs(r) < 0.6 else 'strong')
direction = 'negative' if r < 0 else 'positive'
print(f'Interpretation: {strength} {direction} association')

### 5.5 Youth–Adult Gap by Dominant Employment Sector

In [ ]:
sector_gap = (df[df['Region'].isin(regions_to_show)]
              .groupby('Dom_Sector')['Youth_Gap']
              .agg(['median', 'mean', 'count']).reset_index()
              .rename(columns={'Dom_Sector': 'Sector', 'median': 'Median Gap',
                               'mean': 'Mean Gap', 'count': 'Obs'}))
sector_gap = sector_gap.sort_values('Median Gap', ascending=False)
print(sector_gap.round(2))

fig5, ax5 = plt.subplots(figsize=(7, 4))
colors = {'Agriculture': '#2e7d32', 'Industry': '#1565c0', 'Services': '#e65100'}
bars = ax5.barh(sector_gap['Sector'], sector_gap['Median Gap'],
                color=[colors.get(s, '#999') for s in sector_gap['Sector']],
                alpha=0.85, height=0.5)
for bar, val in zip(bars, sector_gap['Median Gap']):
    ax5.text(val + 0.1, bar.get_y() + bar.get_height() / 2,
             f'{val:.1f} pp', va='center', fontsize=10, fontweight='bold')
ax5.set_xlabel('Median Youth–Adult Gap (percentage points)', fontsize=10)
ax5.set_title('Youth–Adult Unemployment Gap\nby Dominant Employment Sector', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_sector_gap.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 Country Comparison: South Africa vs Germany

In [ ]:
for country in ['South Africa', 'Germany']:
    cdf = df[df['Country'] == country]
    print(f"{country}: avg youth unemp = {cdf['YouthUnemployment'].mean():.1f}%, "
          f"avg youth-adult gap = {cdf['Youth_Gap'].mean():.1f}pp")

da = df[df['Country'] == 'South Africa']
db = df[df['Country'] == 'Germany']

fig6, ax6 = plt.subplots(figsize=(12, 4.5))
ax6.plot(da['Year'], da['YouthUnemployment'], color='#1565c0', lw=2.5,
         marker='o', ms=4, label='South Africa — Youth')
ax6.plot(da['Year'], da['Unemployment Rate'],  color='#1565c0', lw=1.5,
         ls='--', alpha=0.6, label='South Africa — Total')
ax6.plot(db['Year'], db['YouthUnemployment'], color='#c62828', lw=2.5,
         marker='s', ms=4, label='Germany — Youth')
ax6.plot(db['Year'], db['Unemployment Rate'],  color='#c62828', lw=1.5,
         ls='--', alpha=0.6, label='Germany — Total')
ax6.set_xlabel('Year', fontsize=10)
ax6.set_ylabel('Unemployment Rate (%)', fontsize=10)
ax6.set_title('Youth & Total Unemployment: South Africa vs Germany (1991–2022)', fontsize=12, fontweight='bold')
ax6.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig6_country_compare.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6 — Key Findings Summary

| Finding | Detail |
|---|---|
| Persistent youth–adult gap | Youth unemployment averages ~10–12 pp above total unemployment across all regions and years |
| Crisis sensitivity | Both 2008–09 GFC and 2020 COVID caused sharp spikes; youth workers bear a disproportionate share of labour market shocks |
| Regional inequality | Middle East & Africa and Latin America consistently show the highest youth unemployment rates; Asia-Pacific and North America the lowest |
| GDP association | A moderate negative correlation (r ≈ −0.3) between log GDP and youth unemployment confirms wealthier economies tend to have lower youth rates, but GDP alone explains limited variance |
| Sector effect | Countries dominated by the services sector tend to have a larger youth–adult gap; agriculture-dominant economies show a smaller gap, likely due to informal youth employment in agriculture |
| South Africa vs Germany | South Africa's youth unemployment (~55%) is among the world's highest; Germany's (<8%) reflects strong vocational training systems — a stark structural contrast |

## Step 7 — Output: Streamlit App

The analytical workflow above is packaged into an interactive Streamlit application (`app.py`) that allows non-technical users to:

- Filter data by year range and world region
- Compare any two countries side-by-side
- Explore GDP correlations and sector-based analysis
- View KPI summary metrics at a glance

**To run the app locally:**
```bash
pip install -r requirements.txt
streamlit run app.py
```

All data files (`youth_unemployment_global.csv`, `Employment_Unemployment_GDP_data.csv`) must be in the same folder as `app.py`.

---

## Data Citation

- World Bank Open Data. (2025). *Youth unemployment rate (% of total labour force ages 15–24)*. Retrieved April 2025 from https://data.worldbank.org
- International Labour Organization (ILO). (2025). *Employment by sector and unemployment data*. Accessed via Kaggle (World Bank / ILO merged dataset), April 2025.

---
*ACC102 Mini Assignment · Track 4 · Xi'an Jiaotong-Liverpool University · IBSS 2024–25*